# Exercises XP: LoRA Implementation Lab
All `TODO` placeholders have been filled in. Run cells top-to-bottom.

## What you'll learn

- The fundamentals of LoRA (Low-Rank Adaptation) and why it helps churn out efficient fine-tunes.
- How to implement LoRA matrices `A` and `B`, plus how to wrap existing `nn.Linear` layers.
- Differences between standard linear layers, LoRA-enhanced layers, and merged-weight alternatives.
- How to freeze base parameters so that only the LoRA adapters receive updates.

## What you will create

- A reusable `LoRALayer` module and two linear wrappers (`LinearWithLoRA`, `LinearWithLoRAMerged`).
- A 3-layer MLP that can be swapped between standard and LoRA-enhanced variants.
- A minimal MNIST training loop plus accuracy helpers to compare frozen vs. fully-trainable adapters.
- A workflow to freeze baseline weights and fine-tune only the LoRA layers.

# Part 0: Environment Setup

In [ ]:
%pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

In [ ]:
import copy
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

BASE_SEED = 123
torch.manual_seed(BASE_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Exercise 1: Implement `LoRALayer`

Create the low-rank matrices `A` and `B`, scale them with `alpha`, and test the module on a toy tensor.

In [ ]:
class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        std_dev = 1 / torch.sqrt(torch.tensor(rank).float())
        self.A = nn.Parameter(torch.randn(in_dim, rank) * std_dev)
        self.B = nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha

    def forward(self, x):
        # x @ A  -> (batch, rank)
        # (x @ A) @ B -> (batch, out_dim)
        # scale by alpha
        x = self.alpha * (x @ self.A @ self.B)
        return x

# --- Sandbox hyperparameters ---
random_seed = 123
in_dim  = 10   # input features
out_dim = 5    # output features
rank    = 2    # LoRA rank
alpha   = 4    # scaling factor

torch.manual_seed(random_seed)
layer = LoRALayer(in_dim, out_dim, rank, alpha)
x = torch.randn(4, in_dim)   # batch of 4 samples

print("Input x:\n", x)
print("\nLoRALayer:", layer)
print("\nOriginal output:", layer(x))

# Exercise 2: Wrap `nn.Linear` with LoRA

Combine a frozen linear projection plus a trainable `LoRALayer`.

In [ ]:
class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha,
        )

    def forward(self, x):
        # standard linear output + low-rank adaptation
        return self.linear(x) + self.lora(x)

base_linear = nn.Linear(in_dim, out_dim)
layer_lora_1 = LinearWithLoRA(base_linear, rank=rank, alpha=alpha)
print("LinearWithLoRA output:", layer_lora_1(x))

# Exercise 3: Swap a simple network layer with LoRA

Outputs should match before training because `B` is initialised to zeros, making the LoRA term zero.

In [ ]:
class SingleLayerNet(nn.Module):
    def __init__(self, num_features, num_classes):
        super().__init__()
        self.layer = nn.Linear(num_features, num_classes)

    def forward(self, x):
        return self.layer(x)

torch.manual_seed(random_seed)
single_net  = SingleLayerNet(num_features=in_dim, num_classes=out_dim)
sample_input = torch.randn(4, in_dim)

with torch.no_grad():
    baseline_output = single_net(sample_input)

# Replace the plain Linear with a LoRA-wrapped version
single_net.layer = LinearWithLoRA(single_net.layer, rank=rank, alpha=alpha)

with torch.no_grad():
    lora_output = single_net(sample_input)

# B starts at zero => LoRA adds nothing => outputs are identical
print("Outputs match before training?", torch.allclose(baseline_output, lora_output))

# Exercise 4: Merged-weight LoRA layer

Fuse `alpha * A @ B` into the weight matrix so only one `F.linear` call is needed at inference.

In [ ]:
class LinearWithLoRAMerged(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha,
        )

    def forward(self, x):
        lora = self.lora.A @ self.lora.B          # (in_dim, out_dim)
        combined_weight = self.linear.weight + self.lora.alpha * lora.T  # (out_dim, in_dim)
        return F.linear(x, combined_weight, self.linear.bias)

base_linear2 = nn.Linear(in_dim, out_dim)
layer_lora_2 = LinearWithLoRAMerged(base_linear2, rank=rank, alpha=alpha)
print("LinearWithLoRAMerged output:", layer_lora_2(x))

# Exercise 5: Build the MLP and replace layers with LoRA

In [ ]:
class MultilayerPerceptron(nn.Module):
    def __init__(self, num_features, num_hidden_1, num_hidden_2, num_classes):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(num_features, num_hidden_1),  # layer index 0
            nn.ReLU(),                               # layer index 1
            nn.Linear(num_hidden_1, num_hidden_2),  # layer index 2
            nn.ReLU(),                               # layer index 3
            nn.Linear(num_hidden_2, num_classes),   # layer index 4
        )

    def forward(self, x):
        x = self.layers(x)
        return x

# --- Architecture for MNIST ---
num_features = 28 * 28  # 784 pixels per image (flattened)
num_hidden_1 = 128
num_hidden_2 = 64
num_classes  = 10       # digits 0-9

# --- Training settings ---
DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
learning_rate = 5e-4
num_epochs    = 2

torch.manual_seed(BASE_SEED)
model = MultilayerPerceptron(
    num_features=num_features,
    num_hidden_1=num_hidden_1,
    num_hidden_2=num_hidden_2,
    num_classes=num_classes,
)

model.to(DEVICE)
optimizer_pretrained = torch.optim.Adam(model.parameters(), lr=learning_rate)
print(DEVICE)
print(model)
print(optimizer_pretrained)

## Loading dataset

In [ ]:
BATCH_SIZE = 64

# Flatten 28x28 images to vectors of length 784
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda img: img.view(-1)),  # flatten
])

train_dataset = datasets.MNIST(root='data', train=True,  transform=transform, download=True)
test_dataset  = datasets.MNIST(root='data', train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

for images, labels in train_loader:
    print('Image batch dimensions:', images.shape)  # (64, 784)
    print('Image label dimensions:', labels.shape)  # (64,)
    break

## Define evaluation

In [ ]:
def compute_accuracy(model, data_loader, device):
    model.eval()
    correct_pred, num_examples = 0, 0
    with torch.no_grad():
        for features, targets in data_loader:
            features = features.to(device)
            targets  = targets.to(device)
            logits   = model(features)
            _, predicted_labels = torch.max(logits, 1)
            num_examples += targets.size(0)
            correct_pred += (predicted_labels == targets).sum()
    return correct_pred.float() / num_examples * 100

## Training

In [ ]:
def train(num_epochs, model, optimizer, train_loader, device):
    start_time = time.time()
    for epoch in range(num_epochs):
        model.train()
        for batch_idx, (features, targets) in enumerate(train_loader):
            features = features.to(device)
            targets  = targets.to(device)

            logits = model(features)
            loss   = F.cross_entropy(logits, targets)
            optimizer.zero_grad()

            loss.backward()
            optimizer.step()

            if not batch_idx % 400:
                print('Epoch: %03d/%03d|Batch %03d/%03d| Loss: %.4f' %
                      (epoch+1, num_epochs, batch_idx, len(train_loader), loss))

        with torch.set_grad_enabled(False):
            print('Epoch: %03d/%03d training accuracy: %.2f%%' %
                  (epoch+1, num_epochs, compute_accuracy(model, train_loader, device)))

        print('Time elapsed: %.2f min' % ((time.time() - start_time)/60))
    print('Total Training Time: %.2f min' % ((time.time() - start_time)/60))

In [ ]:
train(num_epochs, model, optimizer_pretrained, train_loader, DEVICE)
print(f'Test accuracy: {compute_accuracy(model, test_loader, DEVICE):.2f}%')

# Replacing Linear with LoRA Layers

In [ ]:
model_lora = copy.deepcopy(model)

model_lora.layers[0] = LinearWithLoRAMerged(model_lora.layers[0], rank=4, alpha=8)
model_lora.layers[2] = LinearWithLoRAMerged(model_lora.layers[2], rank=4, alpha=8)
model_lora.layers[4] = LinearWithLoRAMerged(model_lora.layers[4], rank=4, alpha=8)
model_lora.to(DEVICE)

optimizer_lora = torch.optim.Adam(model_lora.parameters(), lr=learning_rate)
print(model_lora)

print(f'Test accuracy orig model: {compute_accuracy(model,      test_loader, DEVICE):.2f}%')
print(f'Test accuracy LoRA model: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

## Exercise 6: Freezing the Original Linear Layers

After calling `freeze_linear_layers`, only the LoRA `A` and `B` parameters will have `requires_grad=True`.

In [ ]:
def freeze_linear_layers(model):
    for child in model.children():
        if isinstance(child, nn.Linear):
            for param in child.parameters():
                param.requires_grad = False
        else:
            # recursively freeze linear layers in children modules
            freeze_linear_layers(child)

freeze_linear_layers(model_lora)
for name, param in model_lora.named_parameters():
    print(f'{name}: {param.requires_grad}')

In [ ]:
# Re-create optimizer so it only tracks the now-trainable LoRA params
optimizer_lora = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_lora.parameters()),
    lr=learning_rate
)

train(num_epochs, model_lora, optimizer_lora, train_loader, DEVICE)
print(f'Test accuracy LoRA finetune: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

print(f'Test accuracy orig model:    {compute_accuracy(model,      test_loader, DEVICE):.2f}%')
print(f'Test accuracy LoRA model:    {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')